In [1]:
## LLM Finetuning

In [2]:
"""
Fine-tuning Qwen for Medical Information Extraction
=====================================================
Model  : Qwen/Qwen2.5-1.5B-Instruct  (small, multilingual, Apache-2.0)
Task   : Structured extraction from clinical / medical free text
Method : QLoRA (4-bit NF4 quantisation + LoRA adapters via PEFT)
Trainer: Hugging Face TRL – SFTTrainer
 
Pip requirements (install once before running):
    pip install torch transformers datasets peft trl bitsandbytes accelerate
 
Optional but recommended:
    pip install wandb          # experiment tracking
    pip install sentencepiece  # some tokenisers need it
"""
 


'\nFine-tuning Qwen for Medical Information Extraction\n=====================================================\nModel  : Qwen/Qwen2.5-1.5B-Instruct  (small, multilingual, Apache-2.0)\nTask   : Structured extraction from clinical / medical free text\nMethod : QLoRA (4-bit NF4 quantisation + LoRA adapters via PEFT)\nTrainer: Hugging Face TRL – SFTTrainer\n \nPip requirements (install once before running):\n    pip install torch transformers datasets peft trl bitsandbytes accelerate\n \nOptional but recommended:\n    pip install wandb          # experiment tracking\n    pip install sentencepiece  # some tokenisers need it\n'

In [4]:
#!pip install peft

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 4.6 MB/s eta 0:00:00


In [5]:
import json
import os
from dataclasses import dataclass, field
from typing import Optional
 
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
 
 

ImportError: cannot import name 'EncoderDecoderCache' from 'transformers' (/home/tatiana.bladier/.local/lib/python3.11/site-packages/transformers/__init__.py)

In [ ]:
# ── 1. Configuration ──────────────────────────────────────────────────────────
@dataclass
class Config:
    # ---- Model ---------------------------------------------------------------
    model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    # Alternative small Qwen multilingual options:
    #   "Qwen/Qwen2.5-0.5B-Instruct"   (~0.5 B params, fastest)
    #   "Qwen/Qwen2.5-3B-Instruct"     (~3  B params, better quality)
 
    # ---- LoRA ----------------------------------------------------------------
    lora_r: int = 16               # rank (8–64; higher = more capacity)
    lora_alpha: int = 32           # scaling factor (typically 2×r)
    lora_dropout: float = 0.05
    lora_target_modules: list = field(
        default_factory=lambda: ["q_proj", "k_proj", "v_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"]
    )
 
    # ---- Quantisation --------------------------------------------------------
    use_4bit: bool = True          # QLoRA; set False if you have >24 GB VRAM
    bnb_4bit_compute_dtype: str = "bfloat16"
    bnb_4bit_quant_type: str = "nf4"
    use_nested_quant: bool = True  # double quantisation
 
    # ---- Training ------------------------------------------------------------
    output_dir: str = "./qwen_medical_extraction"
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4   # effective batch = 8
    learning_rate: float = 2e-4
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    max_seq_length: int = 1024
    logging_steps: int = 10
    save_strategy: str = "epoch"
    fp16: bool = False
    bf16: bool = True              # use False on older GPUs that lack bf16
 
    # ---- Data ----------------------------------------------------------------
    val_size: float = 0.1          # fraction held-out for validation
    seed: int = 42
 
 
CFG = Config()

In [ ]:
# ── 2. Toy Medical Dataset ────────────────────────────────────────────────────
# In production replace this with a real dataset, e.g.:
#   • i2b2 / n2c2 NLP challenges          (https://n2c2.dbmi.hms.harvard.edu/)
#   • MedMentions                          (https://github.com/chanzuckerberg/MedMentions)
#   • MIMIC-III discharge summaries        (requires credentialed access)
#   • datasets.load_dataset("bigbio/...")  (HF BigBIO wrappers)
 
RAW_EXAMPLES = [
    {
        "text": (
            "Patient: Maria Gonzalez, 54-year-old female. "
            "Chief complaint: chest pain radiating to the left arm for 2 hours. "
            "PMH: hypertension, type 2 diabetes mellitus. "
            "Medications: metformin 1000 mg twice daily, lisinopril 10 mg once daily. "
            "Vitals: BP 158/94, HR 102, SpO2 97%. "
            "Diagnosis: NSTEMI."
        ),
        "extraction": {
            "patient_name": "Maria Gonzalez",
            "age": 54,
            "sex": "female",
            "chief_complaint": "chest pain radiating to the left arm for 2 hours",
            "past_medical_history": ["hypertension", "type 2 diabetes mellitus"],
            "medications": [
                {"name": "metformin", "dose": "1000 mg", "frequency": "twice daily"},
                {"name": "lisinopril", "dose": "10 mg", "frequency": "once daily"},
            ],
            "vitals": {"BP": "158/94", "HR": 102, "SpO2": "97%"},
            "diagnosis": "NSTEMI",
        },
    },
    {
        "text": (
            "Patient: Jean-Pierre Dumont, male, 67 years old. "
            "Admitted for acute dyspnea and bilateral leg oedema. "
            "Known heart failure (EF 35%) and chronic kidney disease stage 3. "
            "Current treatment: furosemide 40 mg/day, carvedilol 25 mg twice daily, "
            "spironolactone 25 mg/day. "
            "Labs: creatinine 1.8 mg/dL, BNP 980 pg/mL. "
            "Impression: decompensated heart failure."
        ),
        "extraction": {
            "patient_name": "Jean-Pierre Dumont",
            "age": 67,
            "sex": "male",
            "chief_complaint": "acute dyspnea and bilateral leg oedema",
            "past_medical_history": [
                "heart failure (EF 35%)",
                "chronic kidney disease stage 3",
            ],
            "medications": [
                {"name": "furosemide", "dose": "40 mg", "frequency": "once daily"},
                {"name": "carvedilol", "dose": "25 mg", "frequency": "twice daily"},
                {"name": "spironolactone", "dose": "25 mg", "frequency": "once daily"},
            ],
            "labs": {"creatinine": "1.8 mg/dL", "BNP": "980 pg/mL"},
            "diagnosis": "decompensated heart failure",
        },
    },
    {
        "text": (
            "Patient: Yuki Tanaka, 29-year-old female, presented with rash, "
            "joint pain, and photosensitivity for 3 weeks. "
            "ANA positive (1:320), anti-dsDNA elevated. "
            "No prior significant medical history. "
            "Medications: ibuprofen 400 mg as needed. "
            "Impression: systemic lupus erythematosus (SLE)."
        ),
        "extraction": {
            "patient_name": "Yuki Tanaka",
            "age": 29,
            "sex": "female",
            "chief_complaint": "rash, joint pain, and photosensitivity for 3 weeks",
            "past_medical_history": [],
            "medications": [
                {"name": "ibuprofen", "dose": "400 mg", "frequency": "as needed"}
            ],
            "labs": {"ANA": "positive (1:320)", "anti-dsDNA": "elevated"},
            "diagnosis": "systemic lupus erythematosus (SLE)",
        },
    },
    # ── Add many more examples here for a real run ──
]
 

In [ ]:
# ── 3. Prompt Template ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a clinical NLP assistant. "
    "Given a medical text, extract structured information and return it as valid JSON. "
    "Include all fields present in the text; omit fields that are not mentioned. "
    "Always return ONLY the JSON object — no markdown, no explanation."
)
 
 
def build_chat_prompt(example: dict, tokenizer) -> str:
    """
    Build a single training string in Qwen's ChatML format.
    The model must learn to predict only the assistant turn (the JSON).
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": example["text"]},
        {
            "role": "assistant",
            "content": json.dumps(example["extraction"], ensure_ascii=False, indent=2),
        },
    ]
    # apply_chat_template handles the <|im_start|>/<|im_end|> tokens for Qwen
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

In [ ]:
# ── 4. Dataset Preparation ────────────────────────────────────────────────────
 
def prepare_dataset(raw_examples: list, tokenizer, val_size: float, seed: int) -> DatasetDict:
    """Convert raw examples to a formatted HF DatasetDict."""
    formatted = [
        {"text": build_chat_prompt(ex, tokenizer)} for ex in raw_examples
    ]
    ds = Dataset.from_list(formatted)
    split = ds.train_test_split(test_size=val_size, seed=seed)
    return DatasetDict({"train": split["train"], "validation": split["test"]})

In [ ]:
# ── 5. Model & Tokeniser Loading ──────────────────────────────────────────────
 
def load_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side="right",   # required for SFT with causal LMs
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer
 
 
def load_model(cfg: Config):
    compute_dtype = getattr(torch, cfg.bnb_4bit_compute_dtype)
 
    bnb_config = None
    if cfg.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=cfg.bnb_4bit_quant_type,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=cfg.use_nested_quant,
        )
 
    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        quantization_config=bnb_config,
        device_map="auto",          # spread across available GPUs / CPU
        trust_remote_code=True,
        torch_dtype=compute_dtype if not cfg.use_4bit else None,
    )
    model.config.use_cache = False  # disable KV-cache during training
    return model

def apply_lora(model, cfg: Config):
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.lora_target_modules,
        bias="none",
        inference_mode=False,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model
 

In [ ]:
# ── 6. Training ───────────────────────────────────────────────────────────────
 
def train(cfg: Config):
    print("=" * 60)
    print(f"  Model : {cfg.model_name}")
    print(f"  QLoRA : {cfg.use_4bit}  |  LoRA rank : {cfg.lora_r}")
    print("=" * 60)
 
    # 6-a. Tokeniser
    tokenizer = load_tokenizer(cfg.model_name)
 
    # 6-b. Dataset
    dataset = prepare_dataset(RAW_EXAMPLES, tokenizer, cfg.val_size, cfg.seed)
    print(f"\nDataset — train: {len(dataset['train'])}, val: {len(dataset['validation'])}\n")
 
    # 6-c. Model + LoRA
    model = load_model(cfg)
    model = apply_lora(model, cfg)
 
    # 6-d. Response-only data collator
    #      SFTTrainer + DataCollatorForCompletionOnlyLM masks the prompt tokens
    #      so the loss is computed only on the assistant response (the JSON).
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template,
        tokenizer=tokenizer,
    )
 
    # 6-e. TrainingArguments
    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_train_epochs,
        per_device_train_batch_size=cfg.per_device_train_batch_size,
        gradient_accumulation_steps=cfg.gradient_accumulation_steps,
        learning_rate=cfg.learning_rate,
        lr_scheduler_type=cfg.lr_scheduler_type,
        warmup_ratio=cfg.warmup_ratio,
        weight_decay=cfg.weight_decay,
        fp16=cfg.fp16,
        bf16=cfg.bf16,
        logging_steps=cfg.logging_steps,
        save_strategy=cfg.save_strategy,
        evaluation_strategy=cfg.save_strategy,
        load_best_model_at_end=True,
        report_to="none",           # change to "wandb" if you want tracking
        seed=cfg.seed,
    )
 
    # 6-f. SFTTrainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=collator,
        dataset_text_field="text",
        max_seq_length=cfg.max_seq_length,
        args=training_args,
    )
 
    # 6-g. Train!
    print("\nStarting training …\n")
    trainer.train()
 
    # 6-h. Save final adapter + tokeniser
    final_path = os.path.join(cfg.output_dir, "final_adapter")
    trainer.model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"\n✅ Adapter saved to: {final_path}")
 
    return trainer, tokenizer

In [ ]:
# ── 7. Inference Helper ───────────────────────────────────────────────────────
 
def extract_medical_info(
    text: str,
    model,
    tokenizer,
    max_new_tokens: int = 512,
    device: str = "cuda",
) -> dict:
    """
    Run inference with the fine-tuned model and return a parsed JSON dict.
 
    Usage after training:
        from peft import PeftModel
        base = AutoModelForCausalLM.from_pretrained(CFG.model_name, ...)
        model = PeftModel.from_pretrained(base, "./qwen_medical_extraction/final_adapter")
        result = extract_medical_info(note, model, tokenizer)
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
 
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
 
    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        # Return raw string if model output isn't valid JSON yet
        return {"raw_output": raw_output}
 

In [ ]:
# ── 8. Quick Smoke-Test (CPU, no training) ────────────────────────────────────
 
def smoke_test_prompt(tokenizer):
    """Verify the prompt template renders correctly — no GPU needed."""
    sample = RAW_EXAMPLES[0]
    rendered = build_chat_prompt(sample, tokenizer)
    print("\n── Rendered prompt (first example) ──────────────────────────\n")
    print(rendered)
    print("\n─────────────────────────────────────────────────────────────\n")
 

In [ ]:
# ── 9. Entry Point ────────────────────────────────────────────────────────────
 
if __name__ == "__main__":
    import argparse
 
    parser = argparse.ArgumentParser(description="Fine-tune Qwen for medical IE")
    parser.add_argument(
        "--smoke-test",
        action="store_true",
        help="Only render a sample prompt (no GPU / training required)",
    )
    parser.add_argument(
        "--model",
        default=CFG.model_name,
        help="HF model ID to use (default: %(default)s)",
    )
    parser.add_argument(
        "--epochs", type=int, default=CFG.num_train_epochs,
        help="Number of training epochs",
    )
    parser.add_argument(
        "--no-4bit", action="store_true",
        help="Disable 4-bit quantisation (needs more VRAM)",
    )
    args = parser.parse_args()
 
    CFG.model_name = args.model
    CFG.num_train_epochs = args.epochs
    if args.no_4bit:
        CFG.use_4bit = False
 
    if args.smoke_test:
        # Just load tokeniser and render a prompt — very fast
        tok = load_tokenizer(CFG.model_name)
        smoke_test_prompt(tok)
    else:
        train(CFG)
 